In [ ]:
!pip install -U datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from collections import Counter

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/R-ABNN-LSTM"

os.chdir(PROJECT_DIR)

print(os.getcwd())

/content/drive/MyDrive/R-ABNN-LSTM


In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [24]:
folders = [
    "results/figures"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created.")

Folders created.


In [ ]:
print(dataset)

print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [ ]:
# Build Vocabulary
def tokenize(text):
    return text.lower().split()

counter = Counter()

for sample in dataset["train"]:
    counter.update(tokenize(sample["text"]))

VOCAB_SIZE = 20000

vocab = {
    word:i+2
    for i,(word,_)
    in enumerate(
        counter.most_common(VOCAB_SIZE)
    )
}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

print(len(vocab))

20002


In [26]:
def tokenize(text):
    return text.lower().split()

print(tokenize("I Love Deep Learning"))

['i', 'love', 'deep', 'learning']


In [ ]:
# Encode Reviews
def encode(text):
    return [
        vocab.get(word,1)
        for word in tokenize(text)
    ]

print(
    encode(
        "this movie is great"
    )
)

[10, 20, 7, 88]


In [ ]:
# Padding
MAX_LEN = 200

def pad_sequence(seq):

    if len(seq) > MAX_LEN:
        return seq[:MAX_LEN]

    return seq + [0]*(MAX_LEN-len(seq))

In [28]:
# Process Training Data
train_texts = []
train_labels = []

for sample in dataset["train"]:

    seq = encode(
        sample["text"]
    )

    seq = pad_sequence(seq)

    train_texts.append(seq)

    train_labels.append(
        sample["label"]
    )

In [29]:
# Convert Training Data to Tensors
train_texts = torch.tensor(
    train_texts,
    dtype=torch.long
)

train_labels = torch.tensor(
    train_labels,
    dtype=torch.long
)

In [30]:
# Check Training Shapes
print(train_texts.shape)
print(train_labels.shape)

torch.Size([25000, 200])
torch.Size([25000])


In [31]:
# Save Training Data
torch.save(
    train_texts,
    "checkpoints/train_texts.pt"
)

torch.save(
    train_labels,
    "checkpoints/train_labels.pt"
)

print("Training tensors saved.")

Training tensors saved.


In [32]:
# Process Test Data
test_texts = []
test_labels = []

for sample in dataset["test"]:

    seq = encode(
        sample["text"]
    )

    seq = pad_sequence(seq)

    test_texts.append(seq)

    test_labels.append(
        sample["label"]
    )

In [33]:
# Convert Test Data to Tensors
test_texts = torch.tensor(
    test_texts,
    dtype=torch.long
)

test_labels = torch.tensor(
    test_labels,
    dtype=torch.long
)

In [34]:
# Check Test Shapes
print(test_texts.shape)
print(test_labels.shape)

torch.Size([25000, 200])
torch.Size([25000])


In [35]:
# Save Test Data
torch.save(
    test_texts,
    "checkpoints/test_texts.pt"
)

torch.save(
    test_labels,
    "checkpoints/test_labels.pt"
)

print("Test tensors saved.")

Test tensors saved.


In [36]:
# Verify Saved Files
import os

print(
    os.listdir("checkpoints")
)

['vocab.pt', 'train_dataset.pt', 'test_dataset.pt', 'train_texts.pt', 'train_labels.pt', 'test_texts.pt', 'test_labels.pt']


In [37]:
# Write Experiment Log
log = """
Experiment 1

Dataset: IMDB

Vocabulary Size: 20000

Max Sequence Length: 200

Batch Size: 64

Objective:
Prepare dataset for
LSTM, BN-LSTM and
R-ABNN-LSTM experiments.
"""

with open(
    "thesis_notes/experiment_log.md",
    "a"
) as f:
    f.write(log)

print("Experiment log saved.")

Experiment log saved.
